
# Machine Learning aplicado, Clustering 2
## Pair trading: agrupar acciones para encontrar pares que se mueven juntos

**Curso:** *Machine Learning aplicado* · **Preparado por:** David Díaz, con asistencia de Claude (Anthropic) · **Entorno:** Google Colab

### De dónde venimos

En el notebook anterior agrupamos clientes de un banco: filas que describen personas, y el resultado era un mapa de la
base de clientes. Hoy las filas son **acciones** (las 500 del S&P 500) y el clustering no es el final del análisis sino
un **paso intermedio** dentro de una estrategia de inversión. Es el uso más frecuente del clustering en finanzas: no
para describir, sino para reducir un problema enorme a uno manejable.

La estrategia es el *pair trading*, una de las más antiguas del trading cuantitativo (los equipos de Morgan Stanley la
usaban en los años ochenta), y sirve además para aprender tres cosas que van más allá de ella: cómo describir una
acción con números para que "parecidas" signifique lo que queremos, por qué buscar entre miles de candidatos produce
hallazgos falsos aunque uno no haga nada mal, y cómo se evalúa una estrategia **fuera de muestra** sin engañarse.

### Qué vas a aprender hoy

1. Qué es un par cointegrado, qué es el spread y por qué se puede apostar a que vuelve a su promedio.
2. Por qué probar 125.000 pares a ciegas produce miles de pares "significativos" que no sirven, y cómo el clustering reduce la búsqueda.
3. Dos formas de describir una acción: por su retorno y volatilidad, o por **cómo se mueve junto a las demás** (componentes principales de los retornos). Y por qué la segunda es la que sirve para esto.
4. La prueba de cointegración de Engle y Granger, qué dice su valor p y qué no dice.
5. Un backtest honesto: pares elegidos con los datos de 2021 a 2023 y operados en 2024 y 2025, con parámetros fijos y con parámetros móviles.


In [1]:

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings, time

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from statsmodels.tsa.stattools import coint
import statsmodels.api as sm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 130)
SEMILLA = 0



## 1. El problema: dos acciones que caminan juntas

Coca-Cola y Pepsi venden casi lo mismo a casi la misma gente. Sus precios no son iguales, pero la **razón** entre
ellos se mueve dentro de una banda: cuando una sube mucho más que la otra, suele ser por algo transitorio (una noticia,
un flujo grande), y la diferencia se corrige en semanas. Un *pair trade* apuesta a esa corrección: cuando la acción A
está cara respecto de B, se **vende corto A y se compra B**, y se cierra la posición cuando la diferencia vuelve a su
nivel normal. Si el mercado entero sube o baja, las dos patas se compensan: la apuesta no es sobre el mercado sino
sobre la **relación** entre las dos acciones. Por eso se dice que es una estrategia **neutral al mercado**.

Para que esto funcione hacen falta tres cosas, y cada una tiene su nombre técnico:

- Que la relación entre los dos precios sea **estable en el tiempo**: que exista un número $\beta$ tal que
  $\log P_A - \beta \log P_B$ oscile alrededor de un nivel fijo en vez de irse a cualquier parte. A esa combinación la
  llamamos **spread**, y a la propiedad, **cointegración**.
- Que el spread **vuelva a su promedio** con cierta rapidez (días o semanas, no años). Eso se mide con la **vida media**
  de la reversión.
- Que las desviaciones sean lo bastante grandes para pagar los costos de transar.

El riesgo también tiene nombre: que la relación se rompa. Si una de las dos empresas cambia de negocio, es adquirida o
quiebra, el spread no vuelve nunca y la posición corta pierde sin tope. Por eso toda estrategia de pares lleva un
**stop**: si el spread se aleja demasiado, se cierra y se asume la pérdida.

### Por qué hace falta machine learning

Con 500 acciones hay $500 \cdot 499 / 2 = 124.750$ pares posibles. Probar la cointegración de todos tiene dos
problemas. El primero es de tiempo (cada prueba cuesta una fracción de segundo; son horas). El segundo es más grave y
lo vimos en el curso al hablar de **pruebas múltiples**: si la prueba se equivoca el 5% de las veces cuando no hay
cointegración, sobre 124.750 pares va a declarar cointegrados unos **6.200 pares que no lo están**, y no hay forma de
distinguirlos de los verdaderos mirando solo el valor p. Es el mismo fenómeno de probar mil estrategias y quedarse con
la que "funcionó".

El clustering ataca las dos cosas: agrupa las acciones que se parecen y busca pares **solo dentro de cada grupo**. Hay
menos pruebas, y las que se hacen son entre acciones que tienen una razón económica para moverse juntas. Lo que
tenemos que decidir es qué significa "parecerse", y esa decisión es el corazón del notebook.

### Los datos y la regla de oro

Precios de cierre ajustados de las acciones del S&P 500, diarios, de 2021 a 2025. Y la regla que no se negocia, la
misma del notebook del dólar: los pares se **descubren** con los datos de 2021 a 2023 y se **operan** con los de 2024 y
2025. Si buscamos pares con todos los datos y después "probamos" la estrategia sobre esos mismos datos, el resultado
es tan falso como el Sharpe de 8 del notebook 2.


In [2]:

URL_DATOS = ""        # <-- link al archivo del curso (sp500_precios.csv) si lo tienes; si no, se intenta la copia del curso y después Yahoo Finance
URL_CURSO = "https://raw.githubusercontent.com/daviddiazsolis/clustering_finance_playground/main/notebooks/data/sp500_precios.csv"
URL_SECTORES = "https://raw.githubusercontent.com/daviddiazsolis/clustering_finance_playground/main/notebooks/data/sp500_constituyentes.csv"

def normalizar_link(url):
    if "dropbox.com" in url:
        return url.replace("dl=0", "dl=1").replace("www.dropbox.com", "dl.dropboxusercontent.com")
    if "drive.google.com" in url and "/d/" in url:
        return "https://drive.google.com/uc?export=download&id=" + url.split("/d/")[1].split("/")[0]
    return url

precios, sectores = None, None
for url in [normalizar_link(URL_DATOS)] if URL_DATOS else [URL_CURSO]:
    try:
        precios = pd.read_csv(url, index_col=0, parse_dates=True)
        sectores = pd.read_csv(URL_SECTORES).set_index("ticker")["sector"]
        print("Precios leídos de", url[:60] + "...")
    except Exception as e:
        print("No se pudo leer la copia del curso:", str(e)[:80])
if precios is None:
    import yfinance as yf, io, urllib.request
    html = urllib.request.urlopen(urllib.request.Request("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers={"User-Agent": "Mozilla/5.0"})).read().decode()
    sp = pd.read_html(io.StringIO(html))[0]
    sp["Symbol"] = sp["Symbol"].str.replace(".", "-", regex=False)
    sectores = sp.set_index("Symbol")["GICS Sector"]
    precios = yf.download(sp["Symbol"].tolist(), start="2021-01-01", end="2025-12-31", auto_adjust=True, progress=False)["Close"]
    print("Precios bajados de Yahoo Finance")

print("Días:", len(precios), "  Acciones:", precios.shape[1], "  Desde", precios.index.min().date(), "hasta", precios.index.max().date())


Precios leídos de https://raw.githubusercontent.com/daviddiazsolis/clustering_...
Días: 1254   Acciones: 503   Desde 2021-01-04 hasta 2025-12-30



## 2. Preparar los precios

Algunas acciones entraron al índice hace poco y no tienen historia completa; las sacamos (más de 5% de datos
faltantes). Los huecos sueltos se rellenan con el último precio conocido. Después cortamos en el tiempo: **búsqueda**
hasta diciembre de 2023, **prueba** desde enero de 2024.


In [3]:

precios = precios.loc[:, precios.isna().mean() < 0.05].ffill().dropna()
sectores = sectores.reindex(precios.columns)
busqueda = precios.loc[:"2023-12-31"]
prueba = precios.loc["2024-01-01":]
retornos = busqueda.pct_change().dropna()
print("Acciones con historia completa:", precios.shape[1])
print("Búsqueda:", len(busqueda), "días  |  Prueba:", len(prueba), "días")
print()
print("Acciones por sector:")
print(sectores.value_counts().to_string())


Acciones con historia completa: 489
Búsqueda: 727 días  |  Prueba: 501 días

Acciones por sector:
sector
Industrials               79
Financials                74
Information Technology    72
Health Care               58
Consumer Discretionary    47
Consumer Staples          32
Utilities                 30
Real Estate               30
Materials                 25
Energy                    21
Communication Services    21



## 3. Primer intento: describir cada acción por su retorno y su volatilidad

La forma más simple de describir una acción con números es con su retorno promedio anual y su volatilidad anual en el
período de búsqueda. Dos columnas, 500 filas, y k-means sobre eso. Es lo que hace la mayoría de los tutoriales de
pair trading, y conviene hacerlo primero para ver con los propios ojos qué tiene de malo.


In [4]:

rasgos1 = pd.DataFrame({"retorno": retornos.mean() * 252, "volatilidad": retornos.std() * np.sqrt(252)})
Z1 = StandardScaler().fit_transform(rasgos1)

filas = []
for k in [5, 10, 15, 20, 30, 40]:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(Z1)
    filas.append(dict(K=k, silueta=silhouette_score(Z1, km.labels_), grupo_mas_grande=np.bincount(km.labels_).max()))
print(pd.DataFrame(filas).set_index("K").round(3).to_string())

K1 = 20
km1 = KMeans(n_clusters=K1, n_init=10, random_state=SEMILLA).fit(Z1)
rasgos1["grupo"] = km1.labels_
rasgos1["sector"] = sectores
fig = px.scatter(rasgos1.reset_index(names="ticker"), x="volatilidad", y="retorno", color=rasgos1["grupo"].astype(str), hover_data=["ticker", "sector"],
                 title=f"Las acciones por retorno y volatilidad (2021 a 2023), {K1} grupos de k-means")
fig.update_layout(height=480, width=820, showlegend=False)
fig.show()


    silueta  grupo_mas_grande
K                            
5     0.345               180
10    0.329               121
15    0.337                85
20    0.357                72
30    0.377                42
40    0.371                35


In [16]:
rasgos1

,retorno,volatilidad,grupo,sector
A,0.083542,0.284035,6,Health Care
AAPL,0.165818,0.276098,14,Information Technology
ABBV,0.198594,0.204382,1,Health Care
ABNB,-0.022432,0.511815,2,Consumer Discretionary
ABT,-0.002774,0.226008,0,Health Care
...,...,...,...,...
XYZ,-0.198471,0.663917,4,Financials
YUM,0.114399,0.195330,16,Consumer Discretionary
ZBH,-0.045053,0.255154,0,Health Care
ZBRA,-0.075137,0.397667,10,Information Technology


In [17]:
import random

# Identify tickers in the 'Information Technology' sector
tech_tickers = sectores[sectores == 'Information Technology'].index.tolist()

# Randomly select two tickers from the tech sector
if len(tech_tickers) >= 2:
    selected_tickers = random.sample(tech_tickers, 2)
    ticker1, ticker2 = selected_tickers[0], selected_tickers[1]
    print(f"Selected tickers from Information Technology sector: {ticker1} and {ticker2}")
else:
    print("Not enough tickers in the Information Technology sector to select two.")
    ticker1, ticker2 = None, None

Selected tickers from Information Technology sector: TER and CIEN


Los grupos son franjas en el plano retorno-volatilidad. Nada en ellos dice que dos acciones **se muevan juntas**: un
banco y una farmacéutica con la misma volatilidad y el mismo retorno promedio quedan en el mismo grupo aunque suban y
bajen en días distintos. Tomemos dos empresas al azar del mismo grupo y veamos si se mueven juntas. En promedio deberia funcionar, luego vamos a buscar pares igual, para medir el posible error de solo agrupar por retorno y volatilidad.

In [18]:
# Get the price series for the selected tickers from the 'busqueda' DataFrame
if ticker1 and ticker2:
    series1 = busqueda[ticker1]
    series2 = busqueda[ticker2]

    # Normalize the series (e.g., divide by their first value)
    normalized_series1 = series1 / series1.iloc[0]
    normalized_series2 = series2 / series2.iloc[0]

    # Plot the normalized series
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=normalized_series1.index, y=normalized_series1, mode='lines', name=ticker1))
    fig.add_trace(go.Scatter(x=normalized_series2.index, y=normalized_series2, mode='lines', name=ticker2))

    fig.update_layout(title=f'Normalized Price Series for {ticker1} and {ticker2} (Information Technology)',
                      xaxis_title='Date',
                      yaxis_title='Normalized Price',
                      height=500, width=800)
    fig.show()
else:
    print("Could not plot as tickers were not selected.")

Acabamos de seleccionar aleatoriamente dos empresas del sector de 'Information Technology' (en este caso, **TER** y **CIEN**). Posteriormente, normalizamos sus series de precios dividiendo cada precio por el precio inicial de cada acción. Finalmente, graficamos estas series normalizadas para visualizar si sus movimientos son coordinados o no, es decir, si se mueven más o menos juntas en un eje común.

### La prueba de cointegración

Para cada par dentro de un grupo, la prueba de **Engle y Granger** hace dos cosas: estima por mínimos cuadrados la
relación $\log P_A = \alpha + \beta \log P_B + \varepsilon$, y después pregunta si el residuo $\varepsilon$ (el
spread) es **estacionario**, es decir, si oscila alrededor de un nivel fijo en vez de alejarse sin límite como lo hace
un precio. Si el residuo es estacionario, los dos precios están cointegrados. La función `coint` de statsmodels
devuelve el estadístico y un valor p: con p < 0,05 declaramos el par cointegrado.

Lo que el valor p **no** dice: cuántas veces nos equivocamos al repetir la prueba miles de veces. Esa cuenta la
hacemos nosotros.


In [5]:

log_busqueda = np.log(busqueda)

def pares_cointegrados(grupos, tam_max=60, correlacion_min=None, corr=None):
    filas, pruebas = [], 0
    for g, miembros in grupos.groupby(grupos).groups.items():
        tickers = list(miembros)
        if len(tickers) < 2 or len(tickers) > tam_max:
            continue
        for i in range(len(tickers)):
            for j in range(i + 1, len(tickers)):
                a, b = tickers[i], tickers[j]
                if correlacion_min is not None and corr.loc[a, b] < correlacion_min:
                    continue
                pruebas += 1
                p = coint(log_busqueda[a], log_busqueda[b])[1]
                if p < 0.05:
                    filas.append(dict(grupo=g, a=a, b=b, valor_p=p))
    return pd.DataFrame(filas).sort_values("valor_p").reset_index(drop=True), pruebas

t0 = time.time()
pares1, n_pruebas1 = pares_cointegrados(rasgos1["grupo"])
print(f"Pruebas hechas: {n_pruebas1:,}   pares con p < 0,05: {len(pares1)}   ({len(pares1) / n_pruebas1:.1%} de las pruebas; por puro azar esperaríamos 5%)   [{time.time() - t0:.0f} s]")
print()
pares1["sector_a"] = pares1["a"].map(sectores); pares1["sector_b"] = pares1["b"].map(sectores)
print("Los diez pares con menor valor p:")
print(pares1.head(10).round(5).to_string(index=False))


Pruebas hechas: 5,305   pares con p < 0,05: 443   (8.4% de las pruebas; por puro azar esperaríamos 5%)   [117 s]

Los diez pares con menor valor p:
 grupo    a    b  valor_p               sector_a               sector_b
    18  BBY  IVZ  0.00001 Consumer Discretionary             Financials
    10 EBAY TROW  0.00002 Consumer Discretionary             Financials
     1  KMI  LMT  0.00019                 Energy            Industrials
    10 VTRS ZBRA  0.00037            Health Care Information Technology
    12 CBOE CTAS  0.00040             Financials            Industrials
    12  HIG REGN  0.00041             Financials            Health Care
     7  HST  VST  0.00043            Real Estate              Utilities
     7  HST  PSX  0.00067            Real Estate                 Energy
    16  CMS  DTE  0.00070              Utilities              Utilities
     7 INTU  KKR  0.00080 Information Technology             Financials



Con grupos armados por retorno y volatilidad, el 8% de las pruebas sale "cointegrada" contra un 5% que saldría por
azar puro: la mayoría de los pares encontrados son falsos positivos, y los de menor valor p son parejas sin ninguna
relación económica (una tienda de electrónica con una administradora de fondos, un gasoducto con un fabricante de
armas). Guardamos esta lista para compararla en el backtest. Ahora describamos mejor a las acciones.



## 4. Segundo intento: describir cada acción por cómo se mueve con las demás

Lo que importa para un par es el **co-movimiento**: que los días en que sube una, suba la otra. Esa información está
en la matriz de correlaciones de los retornos, que para 500 acciones tiene 124.750 entradas. Hay que resumirla.

El resumen estándar es el **análisis de componentes principales (PCA)** de los retornos diarios. La idea, en una línea:
buscar las pocas combinaciones de acciones que explican la mayor parte del movimiento conjunto. La primera componente
es siempre "el mercado" (todas las acciones con pesos parecidos y del mismo signo); las siguientes son contrastes
sectoriales (energía contra tecnología, utilities contra consumo discrecional, y así). Cada acción queda descrita por
sus **cargas** en las primeras componentes: diez números que dicen cuánto responde a cada uno de esos movimientos
comunes. Dos acciones con cargas parecidas suben y bajan los mismos días. Eso es exactamente lo que queremos agrupar.
El PCA Playground del hub tiene la geometría completa; aquí lo usamos como herramienta.


In [6]:

Rz = StandardScaler().fit_transform(retornos)              # cada acción con retornos de media 0 y desviación 1
pca = PCA(n_components=10).fit(Rz)
print("Varianza explicada por cada componente:", (pca.explained_variance_ratio_ * 100).round(1), " (suma", round(pca.explained_variance_ratio_.sum() * 100, 1), "%)")
cargas = pd.DataFrame(pca.components_.T, index=retornos.columns, columns=[f"CP{i + 1}" for i in range(10)])
print()
print("Cargas de la primera componente: todas del mismo signo (es el mercado). Signo de las", (cargas["CP1"] > 0).sum(), "cargas positivas de", len(cargas))
print("Cargas de la segunda componente, extremos (es un contraste entre sectores):")
ext = pd.concat([cargas["CP2"].nsmallest(5), cargas["CP2"].nlargest(5)]).to_frame()
ext["sector"] = sectores.reindex(ext.index)
print(ext.round(3).to_string())


Varianza explicada por cada componente: [34.8  6.2  5.3  2.1  2.   1.4  1.2  1.2  1.1  0.9]  (suma 56.1 %)

Cargas de la primera componente: todas del mismo signo (es el mercado). Signo de las 489 cargas positivas de 489
Cargas de la segunda componente, extremos (es un contraste entre sectores):
        CP2                  sector
MPWR -0.082  Information Technology
NVDA -0.081  Information Technology
ON   -0.077  Information Technology
PLTR -0.077  Information Technology
AMD  -0.076  Information Technology
WEC   0.130               Utilities
DUK   0.129               Utilities
ED    0.128               Utilities
LNT   0.126               Utilities
CMS   0.126               Utilities


In [19]:
cargas

,CP1,CP2,CP3,CP4,CP5,CP6,CP7,CP8,CP9,CP10
A,0.050490,-0.014515,-0.055698,0.009766,-0.018375,-0.016298,0.105819,-0.045486,-0.006156,0.004476
AAPL,0.052932,-0.030315,-0.053304,0.022419,-0.047661,0.022643,-0.018468,0.051095,-0.052203,-0.013191
ABBV,0.024294,0.056417,0.001111,-0.016482,-0.071812,0.039957,0.091406,0.053010,-0.033021,0.064682
ABNB,0.042370,-0.072970,-0.010737,0.028818,0.024095,0.090528,-0.055153,0.011980,0.054767,-0.022258
ABT,0.044323,0.022767,-0.061130,-0.017549,-0.031806,0.041785,0.122367,-0.003169,-0.036895,0.006066
...,...,...,...,...,...,...,...,...,...,...
XYZ,0.048283,-0.064106,-0.045182,0.041316,0.025503,0.058561,0.027330,0.002192,0.022913,-0.065310
YUM,0.048739,0.014849,-0.021345,-0.027242,-0.027141,0.048820,-0.044212,0.004499,0.039748,0.030746
ZBH,0.044881,0.003424,-0.004846,-0.008560,-0.004249,0.084767,0.033030,0.002051,0.026900,0.037212
ZBRA,0.051453,-0.043543,-0.027376,0.021025,0.030764,-0.042314,0.023072,0.012968,-0.007173,-0.010798


In [7]:

Z2 = StandardScaler().fit_transform(cargas)
filas = []
for k in [5, 8, 10, 12, 15, 20, 30]:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(Z2)
    filas.append(dict(K=k, silueta=silhouette_score(Z2, km.labels_), grupo_mas_grande=np.bincount(km.labels_).max()))
print(pd.DataFrame(filas).set_index("K").round(3).to_string())

K2 = 12
km2 = KMeans(n_clusters=K2, n_init=10, random_state=SEMILLA).fit(Z2)
grupos2 = pd.Series(km2.labels_, index=retornos.columns, name="grupo")
tabla = pd.crosstab(grupos2, sectores.reindex(grupos2.index))
print()
print("Acciones de cada sector en cada grupo (nadie le dijo al algoritmo qué sector era cada una):")
print(tabla.to_string())


    silueta  grupo_mas_grande
K                            
5     0.219               304
8     0.262               111
10    0.290                97
12    0.293                92
15    0.289                54
20    0.293                49
30    0.261                32

Acciones de cada sector en cada grupo (nadie le dijo al algoritmo qué sector era cada una):
sector  Communication Services  Consumer Discretionary  Consumer Staples  Energy  Financials  Health Care  Industrials  Information Technology  Materials  Real Estate  Utilities
grupo                                                                                                                                                                            
0                            0                       0                20       0           0            0            0                       0          0            0          0
1                            1                       3                 1       0          16           


La tabla es el resultado central del notebook: agrupando por co-movimiento, los grupos **recuperan los sectores**. Hay
un grupo que es casi puro utilities, otro casi puro real estate, otro de bancos, otro de semiconductores. El algoritmo
nunca vio la palabra "sector"; la encontró en los retornos. (Si los grupos coinciden con la clasificación GICS, ¿para
qué el clustering? Porque GICS clasifica por lo que la empresa **declara** que hace y el clustering por cómo **se
comporta**; cuando difieren, el que importa para un par es el segundo.)

### Pares dentro de los grupos, con filtros económicos

Buscamos pares dentro de estos grupos, y esta vez agregamos tres filtros que un operador exigiría antes de mirar
valores p:

- correlación de retornos diarios de al menos 0,6 (que de verdad se muevan juntas);
- $\beta$ entre 0,5 y 2 (que una pata no sea diez veces la otra);
- vida media del spread menor a 60 días hábiles. La vida media se estima con una regresión del cambio del spread
  sobre su nivel rezagado, $\Delta s_t = \gamma\, s_{t-1} + u_t$: si $\gamma < 0$ el spread revierte, y
  $\text{vida media} = -\ln 2 / \gamma$ es cuántos días tarda en recorrer la mitad del camino de vuelta.


In [8]:

correlacion = retornos.corr()

def beta_y_vida_media(a, b):
    ya, yb = log_busqueda[a], log_busqueda[b]
    beta = sm.OLS(ya, sm.add_constant(yb)).fit().params[b]
    s = ya - beta * yb
    ds, rez = s.diff().dropna(), s.shift().dropna()
    gamma = sm.OLS(ds, sm.add_constant(rez.loc[ds.index])).fit().params.iloc[1]
    return beta, (-np.log(2) / gamma if gamma < 0 else np.inf)

t0 = time.time()
pares2, n_pruebas2 = pares_cointegrados(grupos2, tam_max=120, correlacion_min=0.6, corr=correlacion)
print(f"Pruebas hechas: {n_pruebas2:,}   pares con p < 0,05: {len(pares2)}   ({len(pares2) / n_pruebas2:.1%})   [{time.time() - t0:.0f} s]")
pares2[["beta", "vida_media"]] = [beta_y_vida_media(a, b) for a, b in zip(pares2["a"], pares2["b"])]
pares2["correlacion"] = [correlacion.loc[a, b] for a, b in zip(pares2["a"], pares2["b"])]
pares2["sector_a"] = pares2["a"].map(sectores); pares2["sector_b"] = pares2["b"].map(sectores)
candidatos = pares2[(pares2["beta"].between(0.5, 2)) & (pares2["vida_media"] < 60)].sort_values("vida_media").reset_index(drop=True)
print("Pares que pasan los tres filtros:", len(candidatos))
print()
print("Los diez de reversión más rápida:")
print(candidatos.head(10)[["a", "b", "sector_a", "sector_b", "valor_p", "correlacion", "beta", "vida_media"]].round(3).to_string(index=False))


Pruebas hechas: 2,329   pares con p < 0,05: 143   (6.1%)   [49 s]
Pares que pasan los tres filtros: 98

Los diez de reversión más rápida:
   a    b               sector_a               sector_b  valor_p  correlacion  beta  vida_media
 CMS  DTE              Utilities              Utilities    0.001        0.847 0.866      11.915
ACGL   EG             Financials             Financials    0.001        0.793 1.550      12.647
  RF SCHW             Financials             Financials    0.009        0.634 0.561      13.178
AMAT NXPI Information Technology Information Technology    0.001        0.799 1.378      13.616
 AME  ITW            Industrials            Industrials    0.002        0.737 1.025      13.723
INTU  KKR Information Technology             Financials    0.001        0.622 0.962      14.204
LRCX NXPI Information Technology Information Technology    0.003        0.782 1.453      14.241
 IQV   LH            Health Care            Health Care    0.003        0.603 1.018      14.63


Ahora las parejas tienen sentido: dos utilities de Michigan (CMS y DTE), dos constructoras de viviendas (DHI y LEN),
dos petroleras (COP y EOG), dos bancos (BAC y PNC), dos fabricantes de equipos para semiconductores. Esto es lo que
el clustering debía entregar: candidatos con una razón económica para caminar juntos.

## 5. El backtest: operar los pares en 2024 y 2025

La regla de operación es la clásica. Se calcula el spread con el $\beta$ estimado en la búsqueda, se estandariza como
un **z-score** (cuántas desviaciones está de su promedio) y:

- si $z$ cruza hacia arriba de 2, el spread está alto: se vende A y se compra $\beta$ de B;
- si cruza hacia abajo de −2, está bajo: se compra A y se vende $\beta$ de B;
- se cierra cuando $|z| < 0{,}5$ (volvió al promedio) o cuando $|z| > 4$ (stop: la relación se rompió), y no se vuelve a
  abrir hasta que $z$ cruce de nuevo el umbral desde adentro.

El retorno diario de la posición es el del spread, dividido por el capital comprometido en las dos patas
($1 + \beta$) para que sea comparable entre pares. Hay una decisión más: **el promedio y la desviación del spread, ¿se
fijan con los datos de búsqueda o se recalculan con una ventana móvil?** Probamos las dos.

Y una precaución que define el notebook: no vamos a mirar solo los 20 "mejores" pares de cada método, porque
"mejores" se decidió con datos de búsqueda y el ranking puede ser puro ruido. Vamos a mirar también **todos** los pares
que encontró cada método y, como control, **pares elegidos al azar** entre las 489 acciones, operados con la misma regla.
Si los pares al azar rinden igual que los nuestros, lo que estamos midiendo es la regla y el período, no la selección.


In [9]:

log_precios = np.log(precios)

def operar_par(a, b, beta, ventana_movil=None, entrada=2.0, salida=0.5, stop=4.0):
    spread = log_precios[a] - beta * log_precios[b]
    if ventana_movil is None:                                   # parámetros fijos, estimados en la búsqueda
        s_busq = spread.loc[busqueda.index]
        z = (spread - s_busq.mean()) / s_busq.std()
    else:                                                       # parámetros móviles: últimos `ventana_movil` días
        z = (spread - spread.rolling(ventana_movil).mean()) / spread.rolling(ventana_movil).std()
    z = z.loc[prueba.index]
    posicion, actual, previo = pd.Series(0.0, index=z.index), 0, 0.0
    for dia in z.index:
        v = z[dia]
        if actual == 0 and v > entrada and previo <= entrada: actual = -1        # cruza hacia arriba: corto el spread
        elif actual == 0 and v < -entrada and previo >= -entrada: actual = 1    # cruza hacia abajo: largo el spread
        elif actual != 0 and (abs(v) < salida or abs(v) > stop): actual = 0     # volvió, o se rompió
        posicion[dia] = actual
        previo = v
    r_a, r_b = prueba[a].pct_change(), prueba[b].pct_change()
    retorno = (posicion.shift() * (r_a - beta * r_b) / (1 + beta)).fillna(0)
    return retorno, z, posicion

def resumen_cartera(lista, ventana_movil, costo=0.0):
    rets = {}
    for p in lista.itertuples():
        ret, z, pos = operar_par(p.a, p.b, p.beta, ventana_movil)
        rets[f"{p.a}-{p.b}"] = ret - costo * (pos.diff().abs() > 0).astype(float)
    rets = pd.DataFrame(rets)
    total = (1 + rets).prod() - 1
    cartera = rets.mean(axis=1)
    return dict(pares=len(rets.columns), pares_positivos=(total > 0).mean(), mediana_por_par=total.median(), retorno_cartera=(1 + cartera).prod() - 1,
                volatilidad_cartera=cartera.std() * np.sqrt(252), sharpe_cartera=cartera.mean() / cartera.std() * np.sqrt(252)), rets

# las listas a comparar
pares1["beta"] = [beta_y_vida_media(a, b)[0] for a, b in zip(pares1["a"], pares1["b"])]
todos1 = pares1[pares1["beta"].between(0.5, 2)]
rng = np.random.default_rng(SEMILLA)
acciones = list(precios.columns)
azar = pd.DataFrame([(acciones[i], acciones[j]) for i, j in rng.integers(0, len(acciones), (400, 2)) if i != j][:250], columns=["a", "b"])
azar["beta"] = [beta_y_vida_media(a, b)[0] for a, b in zip(azar["a"], azar["b"])]
azar = azar[azar["beta"].between(0.5, 2)]

LISTAS = {"retorno y volatilidad: 20 de menor valor p": todos1.head(20), "retorno y volatilidad: todos los pares": todos1,
          "co-movimiento: 20 de reversión más rápida": candidatos.head(20), "co-movimiento: todos los pares": candidatos,
          "pares al azar (control)": azar}
filas = {}
for nombre, lista in LISTAS.items():
    for vent, etiqueta in [(None, "parámetros fijos"), (60, "ventana móvil de 60 días")]:
        filas[(nombre, etiqueta)], _ = resumen_cartera(lista, vent)
comparacion = pd.DataFrame(filas).T
print("Desempeño fuera de muestra (2024 y 2025), sin costos de transacción:")
print(comparacion.round(3).to_string())


Desempeño fuera de muestra (2024 y 2025), sin costos de transacción:
                                                                     pares  pares_positivos  mediana_por_par  retorno_cartera  volatilidad_cartera  sharpe_cartera
retorno y volatilidad: 20 de menor valor p parámetros fijos           20.0            0.450           -0.011            0.020                0.020           0.498
                                           ventana móvil de 60 días   20.0            0.750            0.061            0.102                0.032           1.526
retorno y volatilidad: todos los pares     parámetros fijos          225.0            0.284           -0.040           -0.022                0.008          -1.332
                                           ventana móvil de 60 días  225.0            0.498           -0.004            0.004                0.022           0.114
co-movimiento: 20 de reversión más rápida  parámetros fijos           20.0            0.350           -0.031        


La tabla hay que leerla de abajo hacia arriba.

**Los pares al azar quedan en cero**: pierden con parámetros fijos y no ganan nada con ventana móvil (la mitad de los
pares termina positivo, la otra mitad negativo). Es el punto de referencia: así rinde la regla cuando la selección no
aporta nada.

**Todos los pares de cada método rinden casi como el azar.** Los pares de retorno y volatilidad y los de co-movimiento
quedan cerca de cero con ventana móvil y en negativo con parámetros fijos. Si la selección hubiera encontrado pares
que de verdad revierten, la cartera completa debería separarse del control, y no lo hace de forma clara.

**Los 20 "mejores" de cada método se ven mejor, y eso es lo peligroso.** Son 20 entre cientos, elegidos por un ranking
que se calculó con los datos de búsqueda; sus resultados tienen mucha más dispersión (mira la volatilidad de cartera)
y la mayor parte de la diferencia con el total es ruido de selección. Un analista que reporte solo esa fila está
haciendo, sin querer, lo mismo que el que prueba mil estrategias y muestra una.

**Los parámetros fijos son peores que los móviles en todas las filas.** El promedio y la desviación del spread de
2021 a 2023 no describen 2024 y 2025: el spread deriva, el z-score se queda meses fuera de rango y la regla abre con
pérdida. Es el problema de todo el curso con otra cara: los parámetros estimados envejecen.

La conclusión honesta, entonces: en 2024 y 2025, con esta regla simple y sin costos, el pair trading sobre el S&P 500
**no deja dinero**, ni con buena selección. No es un fracaso del clustering, que hizo su trabajo (candidatos con sentido
económico y una búsqueda cincuenta veces más chica), sino de la premisa: fueron dos años de tendencias fuertes dentro
de los sectores (semiconductores, bancos, utilities), el peor clima para una estrategia de reversión. Un backtest que
dice "no" con claridad vale más que uno que dice "sí" por haber mirado el futuro.


In [10]:

# Un par de cerca: el spread, el z-score móvil y las posiciones
par = candidatos.iloc[0]
ret, z, pos = operar_par(par.a, par.b, par.beta, ventana_movil=60)
fig = go.Figure()
fig.add_scatter(x=z.index, y=z, name="z-score móvil", line=dict(color="#2563eb"))
for nivel, color in [(2, "#dc2626"), (-2, "#dc2626"), (0.5, "#9ca3af"), (-0.5, "#9ca3af"), (4, "#111827"), (-4, "#111827")]:
    fig.add_hline(y=nivel, line=dict(color=color, dash="dot", width=1))
fig.add_scatter(x=pos.index, y=pos * 3, name="posición (×3 para verla)", line=dict(color="#f59e0b", shape="hv"))
fig.update_layout(title=f"{par.a} contra {par.b} ({par.sector_a}): z-score del spread en 2024 y 2025 y posiciones tomadas", height=420, width=860, yaxis_title="z")
fig.show()
print(f"Retorno acumulado del par en la prueba: {(1 + ret).prod() - 1:.1%}   operaciones (aperturas y cierres): {int((pos.diff().abs() > 0).sum())}")


Retorno acumulado del par en la prueba: 8.1%   operaciones (aperturas y cierres): 26


In [11]:

_, rets_movil = resumen_cartera(candidatos, 60)
acumulado = (1 + rets_movil).cumprod()
fig = go.Figure()
for c in acumulado.columns:
    fig.add_scatter(x=acumulado.index, y=acumulado[c], name=c, line=dict(width=1), opacity=0.35, showlegend=False)
fig.add_scatter(x=acumulado.index, y=(1 + rets_movil.mean(axis=1)).cumprod(), name="cartera de todos los pares por co-movimiento", line=dict(color="black", width=3))
_, rets_azar = resumen_cartera(azar, 60)
fig.add_scatter(x=acumulado.index, y=(1 + rets_azar.mean(axis=1)).cumprod(), name="cartera de pares al azar", line=dict(color="#dc2626", width=3, dash="dash"))
fig.update_layout(title="Capital acumulado por par (gris) y de las carteras equiponderadas (2024 y 2025, ventana móvil, sin costos)", height=480, width=900, yaxis_title="capital (1 = inicio)")
fig.show()



## 6. Lo que hay que llevarse

1. El clustering en finanzas suele ser un paso intermedio: reduce 125.000 candidatos a unos cientos con sentido económico y, de paso, limita el problema de pruebas múltiples.
2. "Parecidas" depende de cómo se describan las acciones. Retorno y volatilidad no dicen nada sobre moverse juntas; las cargas en los componentes principales de los retornos sí, y recuperan los sectores sin que nadie los nombre.
3. La cointegración se prueba con Engle y Granger, pero el valor p no basta: con miles de pruebas hay que exigir además correlación, un $\beta$ razonable y una vida media corta.
4. Fuera de muestra los parámetros envejecen: el spread deriva, y un z-score con media y desviación fijas abre posiciones que nunca cierran. Las ventanas móviles son la corrección mínima.
5. Un backtest honesto se diseña antes de mirar los datos de prueba, compara contra un control al azar, y mira todos los pares y no solo los 20 mejores de un ranking que se hizo con los mismos datos. En 2024 y 2025 la respuesta fue "no", y esa respuesta es información.



## Ejercicios

1. **Sectores declarados contra sectores descubiertos.** Repite la búsqueda de pares usando como grupos directamente el
   sector GICS de cada acción (en vez de los grupos de k-means), con los mismos filtros. ¿Cuántas pruebas se hacen,
   cuántos pares pasan, y cómo rinde la cartera de todos esos pares con ventana móvil? ¿Qué agrega el clustering sobre
   la clasificación oficial?
2. **Costos de transacción.** Cada apertura o cierre de posición cuesta 10 puntos base del capital del par (las dos
   patas). Descuéntalos del retorno diario de cada par y recalcula la fila "co-movimiento: todos los pares" de la
   sección 5. ¿Cuánto cuesta la regla en operaciones por par al año?
3. **La ventana móvil.** Repite el backtest de todos los pares por co-movimiento con ventanas de 20, 40, 60, 120 y
   250 días y grafica el Sharpe de la cartera contra la ventana. ¿Qué pasa con ventanas muy cortas y muy largas, y por
   qué no se puede elegir la mejor mirando esta tabla?



## Soluciones


In [12]:

# Ejercicio 1: grupos = sector GICS
t0 = time.time()
pares_gics, n_gics = pares_cointegrados(sectores.reindex(retornos.columns), tam_max=120, correlacion_min=0.6, corr=correlacion)
pares_gics[["beta", "vida_media"]] = [beta_y_vida_media(a, b) for a, b in zip(pares_gics["a"], pares_gics["b"])]
cand_gics = pares_gics[(pares_gics["beta"].between(0.5, 2)) & (pares_gics["vida_media"] < 60)].sort_values("vida_media").reset_index(drop=True)
res_gics, _ = resumen_cartera(cand_gics, 60)
print(f"Con sectores GICS: {n_gics:,} pruebas, {len(pares_gics)} cointegrados, {len(cand_gics)} pasan los filtros   [{time.time() - t0:.0f} s]")
print(pd.DataFrame({"k-means sobre co-movimiento": comparacion.loc[("co-movimiento: todos los pares", "ventana móvil de 60 días")], "sectores GICS": pd.Series(res_gics)}).round(3).to_string())


Con sectores GICS: 2,335 pruebas, 139 cointegrados, 91 pasan los filtros   [53 s]
                     k-means sobre co-movimiento  sectores GICS
pares                                     98.000         91.000
pares_positivos                            0.469          0.440
mediana_por_par                           -0.011         -0.018
retorno_cartera                           -0.011         -0.008
volatilidad_cartera                        0.019          0.017
sharpe_cartera                            -0.293         -0.217



Los sectores GICS producen una búsqueda del mismo tamaño y una cartera parecida: cuando la clasificación oficial es
buena, el clustering la reproduce y no agrega mucho. Su valor aparece donde no hay clasificación (otros mercados,
activos nuevos, bonos, criptomonedas) o donde la clasificación miente (una empresa clasificada como industrial que se
comporta como tecnológica). El algoritmo no necesita que alguien haya clasificado antes.


In [13]:

# Ejercicio 2: costos de 10 puntos base por apertura o cierre
COSTO = 0.0010
sin, rets_sin = resumen_cartera(candidatos, 60)
con, _ = resumen_cartera(candidatos, 60, costo=COSTO)
ops = np.mean([(operar_par(p.a, p.b, p.beta, 60)[2].diff().abs() > 0).sum() for p in candidatos.itertuples()])
print(f"Operaciones promedio por par en los dos años de prueba: {ops:.1f}  (unas {ops / 2:.0f} por año)")
print(pd.DataFrame({"sin costos": sin, "con 10 pb por operación": con}).loc[["pares_positivos", "retorno_cartera", "sharpe_cartera"]].round(3).to_string())


Operaciones promedio por par en los dos años de prueba: 17.1  (unas 9 por año)
                 sin costos  con 10 pb por operación
pares_positivos       0.469                    0.418
retorno_cartera      -0.011                   -0.028
sharpe_cartera       -0.293                   -0.756



Con ventana móvil la regla opera bastante (una vida media de dos semanas significa entradas y salidas frecuentes), y
diez puntos base por operación se comen más de un punto de retorno al año. Como la cartera ya estaba cerca de cero
antes de costos, después de costos queda claramente en negativo. Por eso los fondos que viven de esto compiten en
costos de ejecución y en velocidad, no en el algoritmo de selección.


In [14]:

# Ejercicio 3: sensibilidad a la ventana móvil
filas = []
for v in [20, 40, 60, 120, 250]:
    r_, _ = resumen_cartera(candidatos, v)
    filas.append(dict(ventana=v, sharpe=r_["sharpe_cartera"], retorno=r_["retorno_cartera"], pares_positivos=r_["pares_positivos"]))
sens = pd.DataFrame(filas).set_index("ventana")
print(sens.round(3).to_string())
fig = px.line(sens.reset_index(), x="ventana", y="sharpe", markers=True, title="Sharpe de la cartera de todos los pares según la ventana del z-score móvil")
fig.update_layout(height=360, width=640)
fig.show()


         sharpe  retorno  pares_positivos
ventana                                  
20       -0.295   -0.010            0.480
40       -0.259   -0.009            0.469
60       -0.293   -0.011            0.469
120      -0.381   -0.014            0.439
250      -0.907   -0.032            0.378



Con todos los pares, ninguna ventana cambia la conclusión: las cortas (20 días) calculan el z-score sobre muy pocos
datos, cualquier movimiento parece "dos desviaciones" y la regla opera de más; las muy largas (250 días) se acercan al
caso de parámetros fijos y vuelve la deriva, y es la que peor rinde. Y la razón por la que no se puede elegir la ventana con esta tabla es la de siempre: la tabla está calculada
sobre 2024 y 2025, que es el período de prueba. Elegir el parámetro que mejor rindió ahí y reportar ese resultado
sería volver a mirar el futuro; la ventana se elige con datos de búsqueda, y la prueba se mira una sola vez.
